# WM-811K baseline·robust 견고성 앙상블 선택

단일 robust 모델은 bias만으로 정상 재현율 보호 조건과 약한 불량 클래스 개선을 동시에 만족하지 못했습니다. 이 노트북은 정상 판별이 안정적인 baseline과 약한 클래스에 강한 robust-flat의 확률을 혼합하고, validation의 clean·4px 이동·1% 활성 die 누락 조건에서 공통 혼합 비율과 none bias를 선택합니다. **test 추론은 수행하지 않습니다.**


In [ ]:
from google.colab import drive, files
from pathlib import Path
import json, shutil, subprocess, sys
import pandas as pd
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'T4 GPU 런타임을 선택한 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))
DATA_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_processed')
EXPERIMENT_ROOT = Path('/content/drive/MyDrive/AI_project/WM811K_robust_hierarchical/validation_multiseed')
OUTPUT_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_robust_hierarchical/validation_stress_ensemble')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for path in (DATA_DIR/'wafer_maps_64.npy', DATA_DIR/'labels.npy', EXPERIMENT_ROOT/'runs'):
    assert path.exists(), f'필수 파일이 없습니다: {path}'


## 로컬 파일 4개를 차례로 선택

1. `train_wm811k_cnn.py`
2. `train_wm811k_robust_hierarchical_candidates.py`
3. `select_wm811k_robust_ensemble.py`
4. `split_assignments.csv`


In [ ]:
def upload_as(target_name):
    print(f'{target_name} 파일을 선택하세요.')
    uploaded = files.upload()
    assert len(uploaded) == 1, '이번 선택창에서는 파일 하나만 선택하세요.'
    _, content = next(iter(uploaded.items()))
    target = Path('/content') / target_name
    target.write_bytes(content)
    return target

BASE_SCRIPT = upload_as('train_wm811k_cnn.py')
ROBUST_SCRIPT = upload_as('train_wm811k_robust_hierarchical_candidates.py')
ENSEMBLE_SCRIPT = upload_as('select_wm811k_robust_ensemble.py')
ASSIGNMENTS = upload_as('split_assignments.csv')


## validation 견고성 앙상블 탐색

각 seed의 baseline과 robust 모델을 세 조건에서 추론한 뒤, robust 혼합 비율과 none bias를 coarse-to-fine 방식으로 탐색합니다. 모든 seed의 정상 재현율·균형 정확도·정확도 보호 기준과 모든 조건의 약한 클래스·macro-F1 개선을 만족해야 통과합니다. 보통 T4에서 약 15~25분 걸립니다.


In [ ]:
command = [
    sys.executable, str(ENSEMBLE_SCRIPT),
    '--maps', str(DATA_DIR/'wafer_maps_64.npy'),
    '--labels', str(DATA_DIR/'labels.npy'),
    '--assignments', str(ASSIGNMENTS),
    '--experiment-root', str(EXPERIMENT_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--batch-size', '512', '--num-workers', '2',
]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
return_code = process.wait()
assert return_code == 0, f'견고성 앙상블 선택 실패: exit code {return_code}'


In [ ]:
summary = json.loads((OUTPUT_DIR/'ensemble_selection_summary.json').read_text(encoding='utf-8'))
display(pd.read_csv(OUTPUT_DIR/'selected_ensemble_seed_metrics.csv'))
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert summary['test_evaluated'] is False
print('앙상블 validation 게이트 통과:', summary['ensemble_validation_gate_passed'])
print('robust 혼합 비율:', summary['selected_robust_weight'])
print('none bias:', summary['selected_none_logit_bias'])
print('배포 모델 변경:', summary['deployment_model_changed'])


## 결과 ZIP 다운로드

다운로드된 `wm811k_robust_ensemble_results.zip`을 프로젝트의 `결과물/wm811k/colab_가져오기/` 폴더에 압축을 풀지 않고 넣으세요.


In [ ]:
EXPORT_DIR = Path('/content/wm811k_robust_ensemble_results')
if EXPORT_DIR.exists(): shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir()
for name in ('ensemble_selection_summary.json', 'selected_ensemble_seed_metrics.csv', 'ensemble_sweep.csv'):
    shutil.copy2(OUTPUT_DIR/name, EXPORT_DIR/name)
plot_path = OUTPUT_DIR/'ensemble_sweep.png'
if plot_path.is_file(): shutil.copy2(plot_path, EXPORT_DIR/plot_path.name)
zip_path = Path(shutil.make_archive(str(EXPORT_DIR), 'zip', root_dir=EXPORT_DIR))
print('다운로드:', zip_path)
files.download(str(zip_path))
